# Week 7 · 01 Transfer learning and domain shift

The CNN you trained in Week 6 learned from 60,000 labelled digits. Here it meets a new dataset (scikit-learn's digits, which look quite different) with only **30** labels. Trained from scratch on 30 images, a CNN gets 71% right; the Week 6 CNN, copied and gently retrained on the same 30, gets 87%. This notebook is about why that works and how to do it in PyTorch: freezing layers, replacing the last layer, choosing learning rates. §2 and §6 are about **domain shift**: a model trained on one kind of data, used on another. Adapting models across domains is the main research topic of Prof. Yoon's lab.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **source / target**: the dataset (or task) a model was first trained on / the new one you want it to work on. Here: MNIST / the sklearn digits.
- **domain**: where the data comes from, and so what it looks like: MNIST's thick handwriting is one domain, sklearn's blurry 8×8 scans another.
- **pretrained**: already trained, on the source. A pretrained model is the starting point, not the end.
- **backbone and head**: the backbone is every layer except the last (it turns an image into features); the head is the last layer (it turns features into class scores).
- **freeze**: keep a layer's weights fixed: `p.requires_grad = False` for each of its parameters, so autograd computes no gradient and the optimizer never changes them.
- **feature extraction**: freeze the backbone, train only a new head.
- **fine-tuning**: start from the pretrained weights and keep training all (or most) of them, with a small learning rate.
- **domain shift**: the target's inputs look different from the source's, so a model that was right on the source makes mistakes on the target.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from figures import *
import checks; checks.load(globals(), "01")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** The notebook in one picture.

In [ ]:
fig_transfer_map()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and booklet 17.3 (layer norm) from memory | warm-up |
| E | the Week 6 CNN, fine-tuned on 30 new images | start from the end: what changed in the weights, and one new digit |
| 1 | why a trained network is worth copying | early layers learn general things |
| 2 | domain shift | why the copied CNN only gets 61% on the new digits |
| 3 | three strategies, in PyTorch | scratch, feature extraction, fine-tuning: `requires_grad`, a new head, the learning rate |
| 4 | how many labels | when copying pays off, and when it stops mattering |
| 5 | why it works | the features, drawn in 2-D |
| 6 | adapting without target labels | domain-adversarial training: the idea behind your lab's AVATAR |
| 7 | the real recipe | a ResNet pretrained on ImageNet, from torchvision |
| 8 | exercises | the block's hand-written parts |

## 0. Recap first · 10 min, closed book

**Quick check:** `nn.Sequential(conv, relu, pool, conv, relu, pool, flatten, linear)`: what is `model[7]`, and what is `model[:7]`?

<details><summary>Answer</summary>

`model[7]` is the last layer (the `Linear`). `model[:7]` is a new `nn.Sequential` of the first seven layers, sharing the same weights: everything up to the flattened features. [Week 6 · 05 §E ↗](../week6/05_cnn_in_pytorch.ipynb)

</details>

**Quick check:** What does `requires_grad=True` on a tensor do?

<details><summary>Answer</summary>

It tells autograd to record the operations on it, so `loss.backward()` fills its `.grad`. Parameters of `nn` layers have it set to `True` automatically. [Week 5 · 06 §2 ↗](../week5/06_pytorch_basics.ipynb)

</details>

**Quick check:** `torch.optim.Adam(model.parameters(), lr=1e-3)`: what does the optimizer do with the list it's given?

<details><summary>Answer</summary>

It keeps that list and, at every `opt.step()`, updates each parameter in it from its `.grad`. A parameter that isn't in the list is never changed by this optimizer. [Week 5 · 05 §7 ↗](../week5/05_optimizers.ipynb)

</details>

**Quick check:** The Week 6 CNN had 9,098 parameters. Where are most of them?

<details><summary>Answer</summary>

In the last layer, `Linear(784, 10)`: $7{,}850$. The two convolutions have $80$ and $1{,}168$. [Week 6 · 05 §3 ↗](../week6/05_cnn_in_pytorch.ipynb)

</details>

**Quick check:** PCA on the rows of $X$: what do the first two scores of an image tell you?

<details><summary>Answer</summary>

Its coordinates along the two directions in which the data varies most: the best 2-D picture of the cloud, in the squared-distance sense. §5 uses it to draw the features. [Week 4 · 02 §6 ↗](../week4/02_pca.ipynb)

</details>

**Exercise 1 · Booklet 17.3: layer norm.** Standardize each **row** of `X` `(n, d)` with its own mean and variance (`axis=1`, `keepdims=True`), then times `gamma` plus `beta` (both `(d,)`).

**Example**: what goes in, and what should come out:
```python
X = np.array([[1.0, 50.0, 0.2], [3.0, 70.0, 0.4]])    # (2, 3) float
layernorm(X, np.ones(3), np.zeros(3)).round(3)
# → array([[-0.69 ,  1.414, -0.724],
#          [-0.666,  1.413, -0.747]])
```

In [ ]:
def layernorm(X, gamma, beta, eps=1e-5):
    pass  # your code

In [ ]:
check(1)

## E. Start from the end: the Week 6 CNN, fine-tuned on 30 new images

`figures.py` gives four helpers this notebook uses: `load_mnist_cnn()` (your Week 6 CNN, read from `../week6/mnist_cnn.pt`), `digits28()` (the target dataset), `few_labels(y, per_class)` (which images get labels) and `accuracy(model, X, y)`. The target is scikit-learn's 1,797 digits, enlarged from 8×8 to fit MNIST's 28×28 format:

In [ ]:
cnn = load_mnist_cnn()                          # the Week 6 CNN: 9,098 weights, 0.972 on the MNIST test set
X, y = digits28()                               # X: (1797, 1, 28, 28) float32 in [0, 1]; y: (1797,) int, the digit
idx, rest = few_labels(y, per_class=3)          # 3 labelled images of each digit: 30. The other 1767 are the test set
print(X.shape, len(idx), len(rest))             # → torch.Size([1797, 1, 28, 28]) 30 1767
print(round(accuracy(cnn, X[rest], y[rest]), 3))   # → 0.607   the CNN as it is: only 61% right

Now **fine-tune** it: copy the CNN and keep training all its weights on the 30 labelled images, with a small learning rate. It's the usual training loop (written out in §3):

In [ ]:
ft = copy.deepcopy(cnn)                         # a separate copy: `cnn` itself stays as it was
ft = train_classifier(ft, X[idx], y[idx], lr=3e-4)       # 100 epochs over the 30 images, Adam, lr 3e-4
print(round(accuracy(ft, X[rest], y[rest]), 3))          # → 0.869

**What the model stores.** Exactly the same 9,098 numbers in the same shapes as before; fine-tuning only moved them. By how much? For each layer, the size of the change relative to the size of the weights:

In [ ]:
for name, before in cnn.state_dict().items():
    after = ft.state_dict()[name]
    print(name, tuple(before.shape), round(((after - before).norm() / before.norm()).item(), 3))
# → 0.weight (8, 1, 3, 3) 0.047
#   0.bias (8,) 0.052
#   3.weight (16, 8, 3, 3) 0.072
#   3.bias (16,) 0.093
#   7.weight (10, 784) 0.198
#   7.bias (10,) 0.324

Read the last column from top to bottom: the first convolution moved by about 5%, the second by 7–9%, the last layer by 20–32%. Fine-tuning kept the general early layers almost as MNIST left them and spent its changes on the specific last layer (§1 explains why that's the right split). **One new image**, one the fine-tuned model never saw:

In [ ]:
img, label = X[rest[17]], y[rest[17]].item()     # img: (1, 28, 28); a 7
with torch.no_grad():
    p_before = torch.softmax(cnn(img[None]), dim=1)[0]   # img[None]: a batch of one, (1, 1, 28, 28)
    p_after = torch.softmax(ft(img[None]), dim=1)[0]
print(label, p_before.argmax().item(), round(p_before.max().item(), 3), p_after.argmax().item(), round(p_after.max().item(), 3))
# → 7 8 0.861 7 0.984

Before: "8" with probability 0.861 (wrong, and confident about it). After: "7" with 0.984 (right). **The rest of this notebook answers:** why are the copied weights such a good start (§1, §5)? Why was the CNN wrong on these images at all (§2)? What else could we have frozen or replaced (§3), and when is it worth it (§4)?

## 1. Why a trained network is worth copying

**The problem.** A CNN has thousands of weights; 30 images can't pin them down. Trained from scratch on our 30 labelled digits, the Week 6 architecture reaches 0.713 on the rest (§4): it overfits, the way Week 6 · 02's networks did with 150 images.

**The idea.** Much of what a network learns isn't specific to its dataset. Its first layer learns **edge detectors**, which are useful for any image of strokes; only the last layer, which turns features into "this is a 7", is really about MNIST. So instead of learning everything again from 30 images, copy the layers that learned general things from 60,000 images, and spend the 30 images on what's specific.

🖼 **Picture.** The first layer of the Week 6 CNN on an MNIST image (top) and on a target image (bottom). The kernels were never shown a sklearn digit, yet they still light up along its strokes: their features carry over.

In [ ]:
fig_conv1_still_works()

**General to specific.** In any deep network trained on images, the layers become more specific with depth: layer 1 finds edges and colour blobs; middle layers find corners, curves and textures; the last layers find whole object parts and the classes. Yosinski et al. (2014, *How transferable are features in deep neural networks?*) measured this: copying the first layers of an ImageNet network helps almost any image task; copying the last layers helps only if the new task is similar.

**Why only this works with depth.** A linear model's weights are tied to its exact input features and classes, so there's nothing general to copy. A deep network's middle layers are a learned **feature extractor**: a function from pixels to useful numbers. That function is the thing that transfers.

**In one sentence:** copy the layers that learned general features on a big dataset, and use your few labels only for what's specific to your task.

## 2. Domain shift: same classes, different look

**The problem.** The target has the same ten classes as MNIST, yet the Week 6 CNN's accuracy falls from 0.972 to 0.607.

🖼 **Picture.** The two domains, one example of each digit. Same classes; thinner or thicker strokes, sharp vs blurry, different proportions.

In [ ]:
fig_two_domains()

**What changed.** Write the data as pairs $(x, y)$ drawn from a distribution. The source draws from $p_S(x, y)$, the target from $p_T(x, y)$. Split each into "what the inputs look like" and "which label goes with an input":

$$
p(x, y) = \underbrace{p(x)}_{\text{what the inputs look like}} \cdot \underbrace{p(y \mid x)}_{\text{the label of an input}}
$$

Here $p(y \mid x)$ is the same in both domains (a 4 is a 4, however it's drawn), but $p_T(x) \neq p_S(x)$: the target's images look different. That case is called **covariate shift** (the inputs, or covariates, shift; the labelling rule doesn't). The network learned its rule only on inputs like the source's; on inputs far from them it has never been checked, and it makes mistakes, as with the 4 read as a 1 in §E.

| Kind of shift | What changes | Example |
|---|---|---|
| covariate shift | $p(x)$; the rule $p(y \mid x)$ stays | MNIST → sklearn digits; one factory's camera → another's; summer → winter sensor readings |
| label shift | $p(y)$, how common each class is | a defect is 1% of parts in one factory, 5% in another |
| concept shift | $p(y \mid x)$ itself | the same reading means "fine" on one machine and "failing" on another |

**How bad can it get? A theorem (stated without proof).** Ben-David et al. (2010) bound a classifier $h$'s error on the target by three terms:

$$
\varepsilon_T(h) \;\le\; \underbrace{\varepsilon_S(h)}_{\text{error on the source}} \;+\; \underbrace{\tfrac{1}{2}\, d_{\mathcal{H}\Delta\mathcal{H}}(S, T)}_{\text{how different the domains look}} \;+\; \underbrace{\lambda}_{\text{the best any } h \text{ can do on both}}
$$

**What it says, in words.** A model can be bad on the target for only three reasons: it's bad on the source; the two domains look different to the kind of model you're using (the middle term is large when some classifier can tell source inputs from target inputs); or no single model is good on both (then no method can help). **Why it matters:** the first term is ordinary training; the third you can't change; so the term to attack is the middle one, by making the two domains look alike to the model. That's what §6's method does, and what most domain-adaptation papers, including your lab's, are built around. (The proof needs learning theory beyond this course.)

**In one sentence:** domain shift means the inputs look different while the labels mean the same; a model is only checked on inputs like its training data, so its accuracy drops by roughly how distinguishable the domains are.

## 3. Three strategies, in PyTorch

🖼 **Picture.** The Week 6 CNN, copied, and the three choices of what may change. Grey = copied and frozen, orange = trains on the 30 target images, white = random start.

In [ ]:
fig_transfer_strategies()

First, the layers by index, since that's how we pick them out of an `nn.Sequential`:

In [ ]:
print(cnn)
# → Sequential(
#     (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
#     (1): ReLU()
#     (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
#     (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
#     (4): ReLU()
#     (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
#     (6): Flatten(start_dim=1, end_dim=-1)
#     (7): Linear(in_features=784, out_features=10, bias=True)
#   )

So the **backbone** is `cnn[:7]` (image → 784 features) and the **head** is `cnn[7]` (784 features → 10 scores). The training loop is the same five lines as always; `figures.py`'s `train_classifier` is exactly this:

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def train_on(model, params, lr, epochs=100):
    torch.manual_seed(1)                                   # the same shuffles for every strategy
    loader = DataLoader(TensorDataset(X[idx], y[idx]), batch_size=50, shuffle=True)   # 30 images: one batch
    opt = torch.optim.Adam(params, lr=lr)                  # updates ONLY the parameters in `params`
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)                  # 1 guess, 2 score
            opt.zero_grad(); loss.backward(); opt.step()   # 3 wipe, 4 blame, 5 adjust
    return model.eval()

**(a) From scratch: the baseline**

In [ ]:
torch.manual_seed(0)
model_a = make_small_cnn()                                 # random weights
model_a = train_on(model_a, model_a.parameters(), lr=1e-3)
print(round(accuracy(model_a, X[rest], y[rest]), 3))       # → 0.713

**(b) Feature extraction: freeze the backbone, train a new head**

**Step 1, freeze.** Set `requires_grad = False` on every parameter. Two things follow: autograd stops computing gradients for them (faster, less memory), and they can't change even if they end up in an optimizer. **Step 2, a new head.** Replace layer 7 with a fresh `nn.Linear(784, 10)`; a new layer's parameters have `requires_grad=True`, so it's the only thing that trains. **Step 3,** give the optimizer only the head's parameters.

In [ ]:
model_b = copy.deepcopy(cnn)
for p in model_b.parameters():
    p.requires_grad = False                                # step 1: freeze all 9,098
torch.manual_seed(0)
model_b[7] = nn.Linear(784, 10)                            # step 2: a new, random head (trainable)
print(sum(p.numel() for p in model_b.parameters() if p.requires_grad))   # → 7850   only the head trains
model_b = train_on(model_b, model_b[7].parameters(), lr=1e-3)            # step 3: the optimizer gets the head only
print(round(accuracy(model_b, X[rest], y[rest]), 3))                     # → 0.791

**Why a new head here, when the classes didn't change?** To show the general recipe: with new classes (say, letters instead of digits) the old head is useless and must be replaced. Here keeping the old head would also have been possible.

**(c) Fine-tuning: train everything, gently**

Start from all the copied weights, train all of them, but with a **smaller learning rate** (3e-4 instead of 1e-3). **Why smaller:** the copied weights are already good; big steps driven by 30 images would overwrite what 60,000 taught (called **catastrophic forgetting**), and the network would overfit like the scratch model. Small steps keep the weights near the good start, which is why §E's convolutions moved only 5–9%.

In [ ]:
model_c = copy.deepcopy(cnn)                               # everything trainable (the default)
model_c = train_on(model_c, model_c.parameters(), lr=3e-4)
print(round(accuracy(model_c, X[rest], y[rest]), 3))       # → 0.869

| Strategy | What trains | Trainable weights | Accuracy (30 labels) |
|---|---|---|---|
| (a) from scratch | everything, from random | 9,098 | 0.713 |
| pretrained, untouched | nothing | 0 | 0.607 |
| (b) feature extraction | a new head | 7,850 | 0.791 |
| (c) fine-tuning | everything, from the copy, lr 3e-4 | 9,098 | 0.869 |

**Remarks.** (1) A common middle way: freeze the first layers (general), fine-tune the later ones (specific). (2) Another: two learning rates, small for the backbone and larger for the new head (exercise 4). (3) **Batch norm** layers in a pretrained network carry running averages from the source; with very few target images, keep them in `eval()` mode while fine-tuning, or recompute them on the target (a cheap domain-adaptation trick called AdaBN).

**In one sentence:** freezing is `requires_grad = False` plus an optimizer that only gets the trainable parameters; fine-tuning trains everything from the copied weights with a small learning rate.

## 4. How many labels make copying worth it?

The same three strategies with 1, 3, 5, 10 and 30 labelled images of each digit (`figures.py`'s `transfer_runs` repeats §3's code for each size):

🖼 **Picture.** Accuracy on the unused sklearn digits against the number of labelled images.

In [ ]:
fig_transfer_curves()

In [ ]:
res = transfer_table()                                     # the numbers behind the picture
for s in res:
    print(f"{s:24s}", res[s])
# → from scratch             [0.598, 0.713, 0.835, 0.896, 0.935]
#   pretrained, no training  [0.608, 0.607, 0.608, 0.606, 0.609]
#   feature extraction       [0.63, 0.791, 0.88, 0.918, 0.959]
#   fine-tuning              [0.872, 0.869, 0.903, 0.933, 0.965]

**Reading it.** (1) With **one** image per digit, fine-tuning gets 0.872 against scratch's 0.598: the copied weights are almost everything. (2) The gap shrinks as labels grow (0.965 vs 0.935 at 300): with enough target data, the target alone can teach the features. (3) Feature extraction trails fine-tuning here because the domains differ: frozen features can't adjust to the blurrier strokes; with a target closer to the source, it would come closer. (4) The untouched model stays at 0.61: no labels, no adaptation (§6 is about adapting without labels).

| Your situation | Usually best |
|---|---|
| few labels, target looks like the source | feature extraction (fast, can't overfit much) |
| few labels, target looks different | fine-tune with a small learning rate, maybe freeze the first layers |
| many labels | fine-tune everything; the start still speeds up training |
| no labels on the target | domain adaptation (§6) |

## 5. Why it works: the features, drawn

The backbone `cnn[:7]` turns each image into 784 numbers. Are those numbers better for telling digits apart than the 784 raw pixels? Draw both with PCA (Week 4: the best 2-D view of a cloud of points):

🖼 **Picture.** The 1,797 target digits, coloured by their label (which neither PCA nor the frozen CNN was shown).

In [ ]:
fig_feature_space()

**A number for the picture.** Train the simplest classifier, logistic regression, on the same 30 labelled images, once on raw pixels and once on the frozen features:

In [ ]:
from sklearn.linear_model import LogisticRegression
with torch.no_grad():
    F_all = cnn[:7](X).numpy()                             # (1797, 784): the frozen features of every image
P_all = X.reshape(len(X), -1).numpy()                      # (1797, 784): the raw pixels
for name, Z in [("pixels", P_all), ("frozen features", F_all)]:
    clf = LogisticRegression(max_iter=2000).fit(Z[idx], y[idx].numpy())
    print(name, round(clf.score(Z[rest], y[rest].numpy()), 3))
# → pixels 0.769
#   frozen features 0.801

The frozen features help even a linear model (0.801 vs 0.769), and it's about what §3's trained head got (0.791): **feature extraction is just a linear classifier on good features**. The features are good because the backbone was trained to make digits linearly separable for its own head; on new images of digits, much of that carries over.

## 6. Adapting without target labels: domain-adversarial training

**The problem.** Often the target has **no** labels at all: a new factory's sensors, a new hospital's scanner. §3 can't be used. But unlabelled target data is usually plentiful.

**The idea** (from §2's theorem). Make the middle term small: learn features in which source and target **look the same**. Test it with a second classifier, the **domain classifier** $D$, which sees a feature vector and guesses "source or target?". If even the best $D$ can't tell, the features don't reveal the domain, and a label classifier trained on source features will work on target features too.

🖼 **Picture.** Domain-adversarial training (DANN). Three networks: a feature extractor $G$, a label classifier $C$ (trained on labelled source images), a domain classifier $D$ (trained on both, to tell the domains apart). Red dashed = gradients.

In [ ]:
fig_dann()

**The objective, piece by piece.** $G$ and $C$ want a small label loss; $D$ wants a small domain loss; $G$ also wants $D$'s loss to be **large**:

$$
\min_{G,\,C}\;\max_{D}\;\; \underbrace{L_{\text{label}}(C(G(x_S)),\, y_S)}_{\text{digits right on the source}} \;-\; \lambda\, \underbrace{L_{\text{domain}}(D(G(x)),\, d)}_{\text{domain guessed right}}
$$

Here $x_S, y_S$ are labelled source images, $x$ are images from both domains with $d$ = 0 for source and 1 for target, and $\lambda > 0$ weighs the two goals. $D$ maximizes the expression, which means minimizing its own loss: it gets good at telling domains apart. $G$ minimizes it, which means **maximizing** $D$'s loss: it hides the domain.

**The trick that makes it one backward pass: the gradient reversal layer (GRL).** Put a layer between $G$ and $D$ that does nothing going forward and multiplies the gradient by $-\lambda$ going backward. Then a single `loss.backward()` on $L_{\text{label}} + L_{\text{domain}}$ gives $D$ its normal gradient (to get better at telling domains apart) and gives $G$ the reversed one (to make that harder). In PyTorch, a layer with a custom backward is an `autograd.Function`:

In [ ]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):              # ctx: a place to keep things for the backward pass
        ctx.lam = lam
        return x.view_as(x)                # forward: the input, unchanged
    @staticmethod
    def backward(ctx, grad):               # grad: the upstream gradient dL/d(output)
        return -ctx.lam * grad, None       # backward: flip and scale it; lam itself gets no gradient

x = torch.tensor([1.0, 2.0], requires_grad=True)
out = GradReverse.apply(x, 0.5)
print(out.detach(), end=" ")
(3 * out).sum().backward()                 # without the layer, d/dx of 3x would be 3
print(x.grad)                              # → tensor([1., 2.]) tensor([-1.5000, -1.5000])

**One training step, as code to read** (not run): `f = G(torch.cat([x_src, x_tgt]))`; `loss_label = ce(C(f[:n_src]), y_src)`; `loss_dom = bce(D(GradReverse.apply(f, lam)), d)`; then `(loss_label + loss_dom).backward()` and one `opt.step()` for all three networks. The label loss uses only the labelled source part; the domain loss uses everything.

**Remarks.** (1) This is Ganin & Lempitsky (2015), the most-cited domain-adaptation method; most adversarial methods since, including your lab's AVATAR (2024), build on it, adding things like self-supervised tasks on the target. Worth knowing well before reading their paper. (2) It's the same min–max game as a GAN (one network tries to fool another), applied to features instead of images. (3) $\lambda$ is usually ramped up from 0 during training, so $G$ first learns useful features before it starts hiding the domain.

**In one sentence:** domain-adversarial training adds a domain classifier and reverses its gradient into the feature extractor, so the features become useful for the task and useless for telling the domains apart.

## 7. The real recipe: a ResNet pretrained on ImageNet

In practice you don't pretrain the source model yourself. torchvision ships networks trained on **ImageNet** (1.28 million photos, 1,000 classes). The recipe is §3's, with a bigger backbone. It's shown here, not run (it downloads 45 MB):

```text
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)              # 11.7 million weights, trained on ImageNet
for p in model.parameters():
    p.requires_grad = False                    # freeze the backbone
model.fc = nn.Linear(model.fc.in_features, 5)  # the head: 512 features -> your 5 classes (trainable)

preprocess = weights.transforms()              # resize to 224 x 224, scale, and normalize with ImageNet's
                                               # mean and std: the target must be fed like the source
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)   # feature extraction
# fine-tuning later: unfreeze, then two learning rates (exercise 4), e.g. 1e-3 for fc, 1e-5 for the rest
```

**Two details that matter.** (1) Here the head is called `model.fc`, not `model[-1]`: read `print(model)` to find a network's last layer. (2) **Preprocess the target exactly like the source** (size, scaling, normalization). Feeding images prepared differently creates a domain shift of your own making.

## 8. Exercises

**Exercise 2 · Freeze, then a new head.** Return a **copy** of `model` (an `nn.Sequential` whose last layer is an `nn.Linear`) with every copied parameter frozen and the last layer replaced by a new `nn.Linear` from the same `in_features` to `n_classes`. The original `model` must not change (use `copy.deepcopy`).

**Example**: what goes in, and what should come out:
```python
model = nn.Sequential(nn.Linear(4, 6), nn.ReLU(), nn.Linear(6, 3))
new = freeze_and_replace_head(model, 5)
new[-1]                                              # → Linear(in_features=6, out_features=5, bias=True)
[p.requires_grad for p in new.parameters()]          # → [False, False, True, True]
```

In [ ]:
def freeze_and_replace_head(model, n_classes):
    pass  # your code

In [ ]:
check(2)

**Exercise 3 · Count what trains.** The number of individual weights with `requires_grad=True` (use `p.numel()`, the number of entries of a tensor).

**Example**: what goes in, and what should come out:
```python
count_trainable(nn.Sequential(nn.Linear(4, 6), nn.ReLU(), nn.Linear(6, 3)))    # → 51   (4·6 + 6 + 6·3 + 3)
```

In [ ]:
def count_trainable(model):
    pass  # your code

In [ ]:
check(3)

**Exercise 4 · Two learning rates.** Return a `torch.optim.Adam` with two **parameter groups**: the last layer's parameters with `lr_head`, all other layers' with `lr_body`. A group is a dict `{"params": ..., "lr": ...}`; pass a list of them to `Adam`. (`model[:-1]` is every layer but the last.)

**Example**: what goes in, and what should come out:
```python
opt = two_lr_optimizer(nn.Sequential(nn.Linear(4, 6), nn.ReLU(), nn.Linear(6, 3)), 1e-3, 1e-5)
[g["lr"] for g in opt.param_groups]     # → [0.001, 1e-05]
```

In [ ]:
def two_lr_optimizer(model, lr_head, lr_body):
    pass  # your code

In [ ]:
check(4)

**Recipe (transfer learning).** (1) Load a model pretrained on a large source. (2) Preprocess the target like the source. (3) Few labels: freeze the backbone (`requires_grad = False`), replace the head, give the optimizer only the head. (4) More labels, or a target that looks different: fine-tune everything from the copy with a small learning rate (or two learning rates). (5) No target labels: domain adaptation (DANN). Always compare with a from-scratch baseline.

In [ ]:
for n in range(1, 5):
    check(n)

---
Solutions: `solutions/01_solutions.py`. Next: `02_autoencoders.ipynb`.